# รัน `interleaved-reasoning` บน Google Colab

Notebook นี้พาไทม์**ต่อจากที่ค้างใน repo** แล้วรันเมทริกซ์การทดลองที่เหลือทั้งหมดจนจบ

**สถานะใน repo (branch `arena/01a0ede9-cot-hybrid`):**

| run | สถานะ |
|---|---|
| `A_mlp_main` (arithmetic) | ✅ done |
| `B_mlp_main` (logic) | ✅ done |
| `C_mlp_main` (recall) | ⏸️ ค้างที่ step 500 → **resume ให้อัตโนมัติ** |
| baselines 6 รัน / ablations 8 / scaling 5 / generalization 11 | ❌ ยังไม่รัน |

**ข้อควรรู้ก่อนเริ่ม**

- โค้ดล็อกไว้รันบน **CPU** (`DEVICE = "cpu"` ใน `scripts/train.py`) → ใช้ runtime **CPU ฟรี** ก็พอ ไม่ต้องเบิร์น GPU quota
- แต่ละรันใช้ ~4–7 นาที (โมเดลเล็ก d=96–128, N=12) → เมทริกซ์ที่เหลือทั้งหมด **~2.5–3.5 ชม.** ต่อเนื่อง
- รันไหนที่ `results/<id>.json` ระบุ `"status": "done"` จะถูก**ข้าม** — ถ้าเซสชันหลุดกลางทาง แค่รันเซลล์ไดรเวอร์ซ้ำก็ได้ต่อทันที
- รันเซลล์ตามลำดับจากบนลงล่างครับ

In [ ]:
# (ทางเลือก) mount Google Drive เพื่อเก็บผลถาวรรอดจากการ reset runtime
# ถ้าไม่ mount ก็ทำงานใน /content ปกติ (ผลอยู่แค่ภายในเซสชัน จนกดดาวน์โหลด/push)
import os
if not os.path.isdir("/content/drive/MyDrive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("mount ไม่สำเร็จ → ใช้ /content ตามปกติ:", e)
BASE = "/content/drive/MyDrive" if os.path.isdir("/content/drive/MyDrive") else "/content"
print("work dir:", BASE)

In [ ]:
# ---- clone repo (branch ของเซสชันนี้) ----
import os, subprocess

REPO_URL = "https://github.com/gunnsmart/cot-hybrid-.git"
BRANCH   = "arena/01a0ede9-cot-hybrid"

os.chdir(BASE)
if not os.path.isdir("cot-hybrid-"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, "cot-hybrid-"], check=True)
else:
    print("มีโฟลเดอร์อยู่แล้ว — ใช้อันเดิม (ถ้าอยากได้ state ล่าสุด ลบโฟลเดอร์แล้วรันเซลล์นี้ใหม่)")
    subprocess.run(["git", "-C", "cot-hybrid-", "fetch", "origin", BRANCH], check=False)

# git identity สำหรับตอน commit (โหมด push / finalize)
subprocess.run(["git", "-C", "cot-hybrid-", "config", "user.name",  "colab-runner"], check=False)
subprocess.run(["git", "-C", "cot-hybrid-", "config", "user.email", "colab-runner@users.noreply.github.com"], check=False)

os.chdir(os.path.join(BASE, "cot-hybrid-", "interleaved-reasoning"))
GIT_FLAG = "--no-git"   # ค่าเริ่มต้น: ไม่ push — เปลี่ยนได้ในเซลล์ "โหมดการ push" ด้านล่าง
print("cwd:", os.getcwd())
!ls

In [ ]:
# ---- dependencies (torch/numpy มีใน Colab อยู่แล้ว, pytest อาจต้องติดตั้ง) ----
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "pytest"], check=False)
import torch, numpy
print("torch", torch.__version__, "| numpy", numpy.__version__, "| cuda:", torch.cuda.is_available(), "(ไม่ใช้ก็ได้ — โค้ดล็อก CPU)")
r = subprocess.run([sys.executable, "-m", "pytest", "--collect-only", "-q"], capture_output=True, text=True)
print("pytest:", (r.stdout.strip().splitlines() or ["(collect failed)"])[-1])

## โหมดการ push (เลือก 1 ใน 2)

กติกาของ repo ([PROBLEM.md §7](interleaved-reasoning/PROBLEM.md)) บังคับ **commit + push ทุก ≤500 steps** — เทรนเนอร์จะ*หยุดเอง*ถ้า push ล้มเหลว บน Colab เลือกได้ว่า:

| โหมด | อย่างไร | เหมาะกับ |
|---|---|---|
| **A — ไม่ push (ง่ายสุด)** | ข้ามเซลล์ถัดไปได้เลย (`--no-git`) | ลองรันดู, รันสั้น ๆ, เซฟผลด้วยการดาวน์โหลด zip ท้าย notebook |
| **B — push ตามกติกา** | วาง GitHub PAT ในเซลล์ถัดไป | รันยาวทั้งเมทริกซ์ กันผลหายถ้าเซสชันหลุด |

PAT ต้องมีสิทธิ์เขียน repo นี้ (fine-grained: **Contents: Read/Write** หรือ classic: scope `repo`) — สร้างที่ github.com → Settings → Developer settings → Personal access tokens

In [ ]:
# ---- โหมด B: ใส่ GitHub PAT (ปล่อยว่าง = ใช้โหมด A ไม่ push) ----
import getpass, subprocess
from urllib.parse import quote

TOKEN = getpass.getpass("GitHub PAT (Enter ว่าง = ไม่ push): ").strip()
if TOKEN:
    url = f"https://{quote(TOKEN, safe='')}@github.com/gunnsmart/cot-hybrid-.git"
    subprocess.run(["git", "remote", "set-url", "origin", url], check=True)
    GIT_FLAG = ""          # เปิด push hook ทุก 100 steps ตาม run_matrix.sh
    print("โหมด B: push ทุก 100 steps เปิดใช้งาน (PAT เก็บใน .git/config ของ VM นี้ — revoke ทิ้งได้เมื่อใช้เสร็จ)")
else:
    GIT_FLAG = "--no-git"
    print("โหมด A: ไม่ push — อย่าลืมเซฟผลในเซลล์ท้าย notebook")

In [ ]:
# ---- smoke test (30 steps, โมเดลจิ๋ว ~ครึ่งนาที) ยืนยันว่าทุกอย่างพร้อม ----
!{sys.executable} -m scripts.train --run-id _smoke --task recall --arch mlp \
    --d 64 --n 4 --depth-max 3 --p-trivial 0.0 --steps 30 --eval-every 30 \
    --batch 64 --train-n 512 --no-git | tail -3
!rm -rf results/_smoke.json checkpoints/_smoke.pt runs/_smoke
print("smoke OK — artefact เก็บกวาดแล้ว")

## รันเมทริกซ์ที่เหลือทั้งหมด (~2.5–3.5 ชม.)

สูตรเดียวกับ `scripts/run_matrix.sh` (recipe จาก calibration `A_cal27`: σ=0.1, gate q=0.30, expectation content) ได้แก่

1. `C_mlp_main` — **resume จาก step 500** (เทรนเนอร์อ่าน step ล่าสุดจาก JSON แล้วโหลด snapshot อัตโนมัติ)
2. baselines 6 รัน (`--force emit` / `--force latent` ครบ 3 tasks) — ใช้ตัดเกณฑ์ 4 (c4)
3. ablations 8 รัน — พิสูจน์ 4 failure modes
4. scaling N=4/8/12/24/48
5. generalization: GRU / SSM / transformer (+ baselines ต่อ arch, cross-task)

รันที่ `status: done` → ข้าม | รันที่ค้าง (`running`) → seed snapshot จาก best checkpoint แล้ว `--resume`

In [ ]:
# ---- matrix driver: รันทุกอันที่ยังไม่เสร็จ ทีละรัน ----
import json, os, shutil, subprocess, sys, time

REC = ("--sigma 0.1 --lam-price 0.2 --lam-commit 0.1 --lam-nd 20 "
       "--reader-layers 2 --batch 96 --q-conf 0.30 --content expectation --push-every 100")

def A(steps=2500, extra=""):
    return f"--task arithmetic --arch mlp --d 128 --n 12 --depth-max 4 --p-trivial 0.0 --steps {steps} {extra}"
def S(n):
    return f"--task arithmetic --arch mlp --d 96 --n {n} --depth-max 4 --p-trivial 0.0 --steps 2000 {REC}"
def G(arch, task="arithmetic", force=""):
    dm = 4 if task == "arithmetic" else 5
    pt = "" if task == "logic" else "--p-trivial 0.0"
    return f"--task {task} --arch {arch} --d 96 --n 12 --depth-max {dm} {pt} --steps 2000 {force} {REC}"

MATRIX = [
    # main — C ค้างอยู่ จะ resume อัตโนมัติ
    ("C_mlp_main", "--task recall --arch mlp --d 128 --n 12 --depth-max 5 --p-trivial 0.0 --steps 3000 " + REC),
    # baselines สำหรับเกณฑ์ 4
    ("A_emit_base",   A(3000, "--force emit")   + " " + REC),
    ("A_latent_base", A(3000, "--force latent") + " " + REC),
    ("B_emit_base",   G("mlp", "logic", "--force emit")),
    ("B_latent_base", G("mlp", "logic", "--force latent")),
    ("C_emit_base",   "--task recall --arch mlp --d 128 --n 12 --depth-max 5 --p-trivial 0.0 --steps 3000 --force emit "   + REC),
    ("C_latent_base", "--task recall --arch mlp --d 128 --n 12 --depth-max 5 --p-trivial 0.0 --steps 3000 --force latent " + REC),
    # ablations (พิสูจน์ 4 failure modes)
    ("A_abl_noprice",   A(2500, "--lam-price 0.0")),
    ("A_abl_nocommit",  A(2500, "--lam-commit 0.0")),
    ("A_abl_nondeg",    A(2500, "--lam-nd 0.0")),
    ("A_abl_bare",      A(2500, "--lam-price 0.0 --lam-commit 0.0 --lam-nd 0.0")),
    ("A_abl_sig0",      A(2500, "--sigma 0.0")),
    ("A_abl_reset",     A(2500, "--semantics reset")),
    ("A_abl_redundant", A(2500, "--p-trivial 0.4")),
    ("A_abl_nogate",    A(2500, "--q-conf 0.0")),
    # scaling N
    ("A_scale_n4", S(4)), ("A_scale_n8", S(8)), ("A_scale_n12", S(12)),
    ("A_scale_n24", S(24)), ("A_scale_n48", S(48)),
    # architecture generalization + per-arch baselines + cross-task
    ("A_gru", G("gru")), ("A_ssm", G("ssm")), ("A_transformer", G("transformer")),
    ("A_gru_emit", G("gru", force="--force emit")), ("A_gru_latent", G("gru", force="--force latent")),
    ("A_ssm_emit", G("ssm", force="--force emit")), ("A_ssm_latent", G("ssm", force="--force latent")),
    ("A_tf_emit",  G("transformer", force="--force emit")), ("A_tf_latent", G("transformer", force="--force latent")),
    ("B_gru", G("gru", "logic")), ("B_transformer", G("transformer", "logic")),
]

def status(rid):
    p = f"results/{rid}.json"
    if os.path.exists(p):
        try:
            return json.load(open(p)).get("status")
        except Exception:
            return None
    return None

T0 = time.time()
failed = []
for rid, extra in MATRIX:
    st = status(rid)
    if st == "done":
        print(f"SKIP   {rid} (done)")
        continue

    resume = ""
    if st == "running":  # ค้างจากคราวก่อน → seed snapshot จาก best checkpoint แล้ว resume
        latest, best = f"runs/{rid}/latest.pt", f"checkpoints/{rid}.pt"
        if not os.path.exists(latest) and os.path.exists(best):
            os.makedirs(f"runs/{rid}", exist_ok=True)
            shutil.copy(best, latest)
            print(f"SEED   {rid}: ทำ snapshot จาก best checkpoint (step ตาม JSON curves)")
        if os.path.exists(f"runs/{rid}/latest.pt"):
            resume = "--resume"

    print(f"\n=== {rid} ===  [{(time.time()-T0)/60:.0f}m elapsed]")
    r = subprocess.run(f"{sys.executable} -m scripts.train --run-id {rid} {extra} {resume} {GIT_FLAG}", shell=True)
    if r.returncode != 0:
        print(f"FAILED {rid} (exit {r.returncode})")
        failed.append(rid)
        if r.returncode == 2:
            print("push ล้มเหลว → หยุดเมทริกซ์ ตามกติกา (แก้ token/remote แล้วรันเซลล์นี้ใหม่ — อันที่ done แล้วจะถูกข้าม)")
            break

print(f"\nจบ: {len(MATRIX) - len(failed)}/{len(MATRIX)} ok | failed: {failed or 'ไม่มี'} | ใช้เวลา {(time.time()-T0)/60:.0f} นาที")

In [ ]:
# ---- ชุดทดสอบ: 6 เกณฑ์ + negative results + ความสม่ำเสมอของตัวเลข ----
# suite จะเขียวได้ก็ต่อเมื่อเมทริกซ์ข้างบนครบแล้ว (c4 ต้องใช้ผล baselines ฯลฯ)
!{sys.executable} -m pytest -q

In [ ]:
# ---- render FINDINGS.md (และบล็อกผลใน README.md) จาก results/*.json ----
!{sys.executable} -m scripts.render_findings
print(open("FINDINGS.md").read())

In [ ]:
# ---- บันทึกผล ----
import os, subprocess

if GIT_FLAG == "":
    # โหมด B: push กลับ GitHub (ระหว่างเทรน push ไปแล้ว ส่วนนี้เก็บงาน FINDINGS/README ที่ render ล่าสุด)
    subprocess.run(["git", "add", "-A"], check=True)
    subprocess.run(["git", "commit", "-m", "colab: matrix complete + FINDINGS rendered", "--allow-empty"], check=False)
    r = subprocess.run(["git", "push", "origin", BRANCH])
    print("push rc =", r.returncode)
else:
    # โหมด A: ดาวน์โหลดเป็น zip (results + checkpoints + เอกสารที่ render แล้ว, ~130MB)
    subprocess.run("zip -qr /content/cot-hybrid-results.zip results checkpoints FINDINGS.md README.md", shell=True, check=True)
    from google.colab import files
    files.download("/content/cot-hybrid-results.zip")

## หมายเหตุ / คำถามที่พบบ่อย

- **เซสชันหลุดกลางทาง?** เปิด Colab ใหม่ → รันเซลล์ mount/clone/deps/token ซ้ำ → กดเซลล์ matrix driver ต่อได้เลย (อัน done จะถูกข้าม อันค้างจะ resume) ในโหมด B ผลทุก 100 steps ถูก push ไว้แล้ว ไม่มีอะไรหาย
- **รายละเอียดการ resume:** ปกติเทรนเนอร์ resume จาก `runs/<id>/latest.pt` (fp16 snapshot — ไฟล์นี้*ไม่*อยู่ใน git) เมื่อไม่มีไฟล์นี้บน Colab ไดรเวอร์จะ seed จาก `checkpoints/<id>.pt` (best-dev ที่ commit ไว้) แทน — weight อาจต่างจาก step สุดท้ายเล็กน้อย ถ้าอยากได้ bit-exact ให้ลบ `results/<id>.json` แล้วรันใหม่ทั้งรัน
- **ทำไมไม่ต้องใช้ GPU:** `scripts/train.py` ล็อก `DEVICE = "cpu"` ไว้เพื่อ reproducibility (noise ของ latent channel ใช้ seeded generator) — โมเดลเล็กมาก จึงเร็วพอบน CPU อยู่แล้ว
- **PAT (โหมด B):** token ถูกฝังใน remote URL ของ `.git/config` ใน VM นี้เท่านั้น ควร revoke หลังใช้งานเสร็จ
- **เช็คความคืบหน้า:** `results/<id>.json` มี curves (loss/dev_acc/mean_p ทุก 250 steps) — เปิดดูได้จาก file browser ซ้ายมือ หรือดู log ของเทรนเนอร์ที่พิมพ์ทุก 200 steps